# Streamlit Submission BFGAI - Muslichin

Notebook ini membuat file `logic.py` dan `app.py`, lalu menjalankan aplikasi Streamlit untuk text-to-image, batch generation, scheduler switching, inpainting, dan outpainting/zoom-out.

# Prepare Dependencies

In [ ]:
!pip install -q pyngrok streamlit torch diffusers transformers accelerate Pillow numpy streamlit_drawable_canvas==0.8.0

In [ ]:
from pyngrok import ngrok
import subprocess

# Streamlit

## logic.py

In [ ]:
%%writefile logic.py
import gc

import torch
from diffusers import (
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler,
    StableDiffusionInpaintPipeline,
    StableDiffusionPipeline,
)
from PIL import Image, ImageFilter

BASE_MODEL_ID = "runwayml/stable-diffusion-v1-5"
INPAINT_MODEL_ID = "runwayml/stable-diffusion-inpainting"


def _device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def _dtype():
    return torch.float16 if torch.cuda.is_available() else torch.float32


def _generator(seed):
    return torch.Generator(device=_device()).manual_seed(int(seed))


@torch.inference_mode()
def load_models_cached():
    device = _device()
    dtype = _dtype()
    print(f"Loading models to {device}")

    pipe_txt2img = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(device)
    pipe_txt2img.enable_attention_slicing()
    pipe_txt2img.enable_vae_slicing()

    pipe_inpaint = StableDiffusionInpaintPipeline.from_pretrained(
        INPAINT_MODEL_ID,
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(device)
    pipe_inpaint.enable_attention_slicing()
    pipe_inpaint.enable_vae_slicing()

    return pipe_txt2img, pipe_inpaint


def flush_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Memory Flushed!")


def set_scheduler(pipe, scheduler_name):
    if scheduler_name == "Euler A":
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "DPM++":
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
    elif scheduler_name == "DDIM":
        pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    else:
        raise ValueError("scheduler_name must be one of: Euler A, DPM++, DDIM")
    return pipe


@torch.inference_mode()
def generate_image(pipe, prompt, neg_prompt, seed, steps, cfg, num_images=1, scheduler_name="Euler A"):
    pipe = set_scheduler(pipe, scheduler_name)
    num_images = max(1, min(int(num_images), 4))
    generators = [_generator(int(seed) + idx) for idx in range(num_images)]

    result = pipe(
        prompt=[prompt] * num_images,
        negative_prompt=[neg_prompt] * num_images,
        generator=generators,
        num_inference_steps=int(steps),
        guidance_scale=float(cfg),
    )
    return result.images


@torch.inference_mode()
def run_inpainting(pipe, image, mask, prompt, strength):
    if image.mode != "RGB":
        image = image.convert("RGB")
    if mask.mode != "L":
        mask = mask.convert("L")
    if image.size != mask.size:
        mask = mask.resize(image.size, resample=Image.NEAREST)

    result = pipe(
        prompt=prompt,
        image=image,
        mask_image=mask,
        strength=float(strength),
        guidance_scale=8.0,
        num_inference_steps=35,
        generator=_generator(9),
    )
    return result.images[0]


def prepare_outpainting(image, expand_pixels=128):
    if image.mode != "RGB":
        image = image.convert("RGB")

    width, height = image.size
    new_width = width + (expand_pixels * 2)
    new_height = height + (expand_pixels * 2)
    new_width -= new_width % 8
    new_height -= new_height % 8

    bg = image.resize((new_width, new_height), resample=Image.BICUBIC)
    bg = bg.filter(ImageFilter.GaussianBlur(radius=50))

    canvas = bg.copy()
    paste_x = (new_width - width) // 2
    paste_y = (new_height - height) // 2
    canvas.paste(image, (paste_x, paste_y))

    mask = Image.new("L", (new_width, new_height), 255)
    inner_box = Image.new("L", (width, height), 0)
    mask.paste(inner_box, (paste_x, paste_y))

    return canvas, mask

## app.py

In [ ]:
%%writefile app.py
import numpy as np
import streamlit as st
from PIL import Image, ImageFilter
from streamlit_drawable_canvas import st_canvas

import logic

st.set_page_config(page_title="StudioAI", layout="wide", page_icon="AI")


@st.cache_resource
def get_models():
    return logic.load_models_cached()


try:
    pipe_txt2img, pipe_inpaint = get_models()
except Exception as exc:
    st.error(f"Error loading models: {exc}")
    st.stop()

st.title("StudioAI: Stable Diffusion Image Suite")

with st.sidebar:
    st.header("Parameters")
    steps = st.slider("Quality Steps", 15, 50, 30)
    cfg = st.slider("Creativity (CFG)", 1.0, 20.0, 7.5)
    seed = st.number_input("Seed Control", value=222, step=1)

    st.divider()
    st.subheader("Generation")
    scheduler_name = st.selectbox("Scheduler", ["Euler A", "DPM++", "DDIM"])
    num_images = st.slider("Batch Size", 1, 4, 1)

    st.divider()
    if st.button("Clear Memory"):
        logic.flush_memory()
        st.toast("Memory cleared")

tab_gen, tab_edit = st.tabs(["Generate", "Edit"])

with tab_gen:
    c1, c2 = st.columns([1, 1], gap="large")

    with c1:
        st.subheader("Input")
        with st.form(key="gen_form"):
            prompt = st.text_area(
                "Prompt",
                "a dreamy hand drawn sci-fi illustration of a tiny astronaut floating near a colorful planet",
                height=150,
            )
            neg_prompt = st.text_input(
                "Negative Prompt",
                "photorealistic, realistic, photograph, 3d render, messy, blurry, low quality, bad art, ugly",
            )
            submit_gen = st.form_submit_button("Generate", type="primary")

        if submit_gen:
            with st.spinner("Generating image"):
                logic.flush_memory()
                generated_list = logic.generate_image(
                    pipe_txt2img,
                    prompt,
                    neg_prompt,
                    seed,
                    steps,
                    cfg,
                    num_images,
                    scheduler_name,
                )
                st.session_state["generated_images"] = generated_list
                if generated_list:
                    st.session_state["current_image"] = generated_list[0]
            st.rerun()

    with c2:
        st.subheader("Output")
        if "generated_images" in st.session_state:
            imgs = st.session_state["generated_images"]
            if len(imgs) > 1:
                cols = st.columns(2)
                for idx, img in enumerate(imgs):
                    with cols[idx % 2]:
                        st.image(img, caption=f"Image {idx + 1}", use_container_width=True)
                        if st.button(f"Select Image {idx + 1}", key=f"sel_{idx}"):
                            st.session_state["current_image"] = img
                            st.toast(f"Image {idx + 1} selected")
            elif len(imgs) == 1:
                st.image(imgs[0], caption="Result", use_container_width=True)
        else:
            st.info("Enter a prompt and press Generate.")

with tab_edit:
    if "current_image" not in st.session_state:
        st.info("Generate an image first.")
    else:
        source_img = st.session_state["current_image"]
        if isinstance(source_img, list):
            source_img = source_img[0]
            st.session_state["current_image"] = source_img

        width, height = source_img.size
        mode = st.radio("Mode", ["Inpainting", "Outpainting Zoom Out"], horizontal=True)
        st.divider()

        if mode == "Inpainting":
            col_tools, col_result = st.columns([1, 1], gap="large")
            if "canvas_key" not in st.session_state:
                st.session_state["canvas_key"] = 0

            dynamic_key = f"canvas_{st.session_state['canvas_key']}_{id(source_img)}"

            with col_tools:
                st.subheader("Draw Mask")
                canvas_result = st_canvas(
                    fill_color="rgba(255, 255, 255, 1.0)",
                    stroke_width=20,
                    stroke_color="#FFFFFF",
                    background_image=source_img,
                    update_streamlit=True,
                    height=height,
                    width=width,
                    drawing_mode="freedraw",
                    key=dynamic_key,
                )

            with col_result:
                st.subheader("Settings")
                with st.form("inpaint_input"):
                    edit_prompt = st.text_input("Edit Prompt", "a broken satellite floating in space")
                    strength = st.slider("Strength", 0.1, 1.0, 0.85)
                    submit_inpaint = st.form_submit_button("Run Inpainting", type="primary")

                if submit_inpaint:
                    if canvas_result.image_data is None or np.max(canvas_result.image_data) == 0:
                        st.warning("Draw a mask first.")
                    else:
                        with st.spinner("Processing inpainting"):
                            logic.flush_memory()
                            mask_data = canvas_result.image_data[:, :, 3]
                            mask_data[mask_data > 0] = 255
                            mask_image = Image.fromarray(mask_data.astype("uint8"), mode="L")
                            if mask_image.size != source_img.size:
                                mask_image = mask_image.resize(source_img.size, resample=Image.NEAREST)
                            mask_image = mask_image.filter(ImageFilter.MaxFilter(15))
                            result_img = logic.run_inpainting(pipe_inpaint, source_img, mask_image, edit_prompt, strength)
                            st.session_state["current_image"] = result_img
                            st.session_state["canvas_key"] += 1
                        st.rerun()

        else:
            col_original, col_action = st.columns([1, 1], gap="large")
            with col_original:
                st.subheader("Original")
                st.image(source_img, caption="Current image", use_container_width=True)
            with col_action:
                st.subheader("Expansion")
                with st.form("outpaint_input"):
                    out_prompt = st.text_input(
                        "Full Image Prompt",
                        "wide angle view of the same illustrated sci-fi astronaut scene, detailed background",
                    )
                    submit_outpaint = st.form_submit_button("Zoom Out", type="primary")

                if submit_outpaint:
                    with st.spinner("Expanding canvas"):
                        logic.flush_memory()
                        canvas_ready, mask_ready = logic.prepare_outpainting(source_img)
                        final_result = logic.run_inpainting(pipe_inpaint, canvas_ready, mask_ready, out_prompt, 1.0)
                        st.session_state["current_image"] = final_result
                    st.rerun()

# Menggunakan Ngrok Untuk Deployment

## Konfigurasi Autentikasi Ngrok dan Menjalankan Aplikasi Streamlit

Isi `auth_token` dengan token Ngrok pribadi sebelum menjalankan cell ini di Colab.

In [ ]:
auth_token = "YOUR_AUTHENTICATION_KEY"

ngrok.set_auth_token(auth_token)
subprocess.Popen(["streamlit", "run", "app.py"])

## Membuat Public URL

In [ ]:
public_url = ngrok.connect(8501).public_url
print(public_url)

## Menutup Semua Tunnel Ngrok yang Aktif

In [ ]:
ngrok.kill()
print("All active ngrok tunnels have been closed.")